# EA-TTM — TTM Model Contract and Interface Audit

**Project:** Extreme-Aware Tiny Time Mixers (EA-TTM) for Retail Demand Forecasting under Regime Shifts  
**Phase:** 01 — TTM model contract and interface audit  
**Scope:** Resolve the exact pre-trained TTM model, freeze its provenance, inspect its internal module structure, and empirically identify the tensor interfaces required by later head probing and EA-TTM decoder cross-feed.

This phase performs **no target-domain training**.

## Locked model-selection contract

The approved project settings are:

- context length: 512 daily observations;
- forecast horizon: 28 daily observations;
- TTM repository: `ibm-granite/granite-timeseries-ttm-r2`;
- frequency: daily (`D`);
- loss preference: standard MSE-trained TTM;
- project seed: 2026.

The selector is allowed to choose the compatible R2.1 revision, but the exact selected revision and Hugging Face commit SHA are recorded and then frozen for the rest of the project.


## Experimental model lineage

The project distinguishes three complete forecasting systems:

$$
\mathcal{M}_{ZS}
\;\longrightarrow\;
\mathcal{M}_{HP}
\;\longrightarrow\;
\mathcal{M}_{EA}
$$

where:

- $\mathcal{M}_{ZS}$ is the zero-shot pre-trained TTM;
- $\mathcal{M}_{HP}$ is obtained by freezing the TTM backbone and training the Slim TTM Decoder plus Forecasting Head on the retail domain;
- $\mathcal{M}_{EA}$ freezes the completed head-probed TTM and trains only the shock-aware residual adapter in the primary EA-TTM experiment.

The final EA-TTM forecast will later be defined as:

$$
\hat{y}_{EA,h}
=
\hat{y}_{HP,h}
+
g_h\,\Delta_h
$$

The purpose of this audit is to determine the **actual implementation-level tensors** that correspond to the backbone representation, Slim Decoder representation, forecasting-head input, and base forecast. No tensor shape is assumed in advance.


## Interface-audit principle

Let the selected TTM receive a univariate batch tensor

$$
\mathbf{X}
\in
\mathbb{R}^{B\times L\times C}
$$

with:

- $B=1$ synthetic audit sample;
- $L=512$ historical time steps;
- $C=1$ target channel.

The model must return a point forecast with 28 requested horizon steps. Internal tensors are discovered through:

1. the model output object;
2. the complete named-module inventory;
3. forward hooks on modules whose implementation names indicate backbone, decoder, or forecasting-head functionality.

The audit records observed shapes rather than imposing architecture assumptions.


## Runtime dependency preflight

Colab runtimes are ephemeral. Therefore, each execution-critical notebook validates its own import surface rather than assuming that packages installed in a previous session remain available.

The project requires the public IBM TSFM source namespace:

```text
tsfm_public
```

The preflight follows this order:

1. verify/install the pinned `granite-tsfm==0.3.9` distribution without replacing the active PyTorch build;
2. test whether `tsfm_public` is importable;
3. if the installed distribution does not expose the required source namespace, attach the official IBM repository source at a frozen Git commit;
4. record the exact source commit used.

The fallback source commit is persisted in project metadata so later Colab sessions use the same code revision rather than silently following a moving branch.


In [1]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive

# Google Drive is the persistent project state for all Colab sessions.
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

TSFM_DISTRIBUTION_VERSION = "0.3.9"
TSFM_GIT_REPOSITORY = "https://github.com/ibm-granite/granite-tsfm.git"
TSFM_GIT_TAG = f"v{TSFM_DISTRIBUTION_VERSION}"

RUNTIME_VENDOR_ROOT = Path("/content/ea_ttm_vendor")
TSFM_SOURCE_ROOT = RUNTIME_VENDOR_ROOT / "granite-tsfm"
TSFM_SOURCE_LOCK = PROJECT_ROOT / "metadata" / "00_TSFM_SOURCE_LOCK.txt"

RUNTIME_VENDOR_ROOT.mkdir(parents=True, exist_ok=True)
TSFM_SOURCE_LOCK.parent.mkdir(parents=True, exist_ok=True)


def run_command(command: list[str], cwd: Path | None = None) -> str:
    """Run a subprocess command and return stripped stdout."""
    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=True,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()


def module_available(module_name: str) -> bool:
    """Return True only when Python can resolve the requested module."""
    return importlib.util.find_spec(module_name) is not None


Mounted at /content/drive


In [2]:
# Install the pinned distribution only if it is not already present.
# --no-deps avoids replacing the Colab PyTorch/CUDA build captured in Phase 00.
try:
    from importlib.metadata import PackageNotFoundError, version

    installed_tsfm_version = version("granite-tsfm")
except PackageNotFoundError:
    installed_tsfm_version = None

if installed_tsfm_version != TSFM_DISTRIBUTION_VERSION:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            f"granite-tsfm=={TSFM_DISTRIBUTION_VERSION}",
        ],
        check=True,
    )

# Ensure lightweight runtime dependencies required by the IBM utilities exist.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "datasets",
        "deprecated",
        "einops>=0.7",
        "pyyaml>=6.0",
        "huggingface-hub",
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'datasets', 'deprecated', 'einops>=0.7', 'pyyaml>=6.0', 'huggingface-hub'], returncode=0)

In [3]:
def resolve_tsfm_source_commit() -> tuple[str, str]:
    """
    Resolve a stable IBM TSFM source revision.

    Preference:
    1. reuse the commit already locked by this project;
    2. use Git tag v0.3.9 if IBM exposes it;
    3. otherwise freeze the current official main-branch commit once.
    """
    if TSFM_SOURCE_LOCK.exists():
        locked_commit = TSFM_SOURCE_LOCK.read_text(encoding="utf-8").strip()
        if locked_commit:
            return locked_commit, "project_lock"

    tag_query = subprocess.run(
        [
            "git",
            "ls-remote",
            "--tags",
            TSFM_GIT_REPOSITORY,
            f"refs/tags/{TSFM_GIT_TAG}",
        ],
        check=False,
        capture_output=True,
        text=True,
    ).stdout.strip()

    if tag_query:
        commit = tag_query.split()[0]
        source = TSFM_GIT_TAG
    else:
        head_query = run_command(
            [
                "git",
                "ls-remote",
                TSFM_GIT_REPOSITORY,
                "refs/heads/main",
            ]
        )
        commit = head_query.split()[0]
        source = "main_frozen_on_first_use"

    TSFM_SOURCE_LOCK.write_text(commit, encoding="utf-8")
    return commit, source


tsfm_source_commit = None
tsfm_source_resolution = "installed_distribution"

if not module_available("tsfm_public"):
    tsfm_source_commit, tsfm_source_resolution = resolve_tsfm_source_commit()

    if TSFM_SOURCE_ROOT.exists():
        subprocess.run(
            ["rm", "-rf", str(TSFM_SOURCE_ROOT)],
            check=True,
        )

    run_command(
        ["git", "clone", "--quiet", "--no-checkout", TSFM_GIT_REPOSITORY, str(TSFM_SOURCE_ROOT)]
    )
    run_command(
        ["git", "checkout", "--quiet", tsfm_source_commit],
        cwd=TSFM_SOURCE_ROOT,
    )

    sys.path.insert(0, str(TSFM_SOURCE_ROOT))

assert module_available("tsfm_public"), (
    "IBM TSFM source namespace `tsfm_public` is still unavailable after "
    "the runtime preflight."
)

print("TSFM import surface: READY")
print("TSFM source resolution:", tsfm_source_resolution)
print("TSFM source commit:", tsfm_source_commit)


TSFM import surface: READY
TSFM source resolution: installed_distribution
TSFM source commit: None


In [4]:
from __future__ import annotations

import copy
import hashlib
import inspect
import json
import os
import platform
import random
import sys
from dataclasses import asdict, is_dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from huggingface_hub import HfApi
from tsfm_public.toolkit.get_model import get_model
from tsfm_public.toolkit.time_series_preprocessor import DEFAULT_FREQUENCY_MAPPING


## Project and model constants

All model-selection inputs are explicit constants. The exact model revision is deliberately **not** hard-coded; it is first resolved by the official TTM selector and then written into the project metadata.


In [5]:
PROJECT_ROOT = Path("/content/drive/MyDrive/EA_TTM_RETAIL")

BASE_SEED = 2026
TTM_REPOSITORY = "ibm-granite/granite-timeseries-ttm-r2"
REQUESTED_CONTEXT_LENGTH = 512
REQUESTED_PREDICTION_LENGTH = 28
DATA_FREQUENCY = "D"
PREFER_L1_LOSS = False

MODEL_SNAPSHOT_DIR = PROJECT_ROOT / "models" / "pretrained_ttm"
METADATA_DIR = PROJECT_ROOT / "metadata"
RESULTS_DIR = PROJECT_ROOT / "results" / "tables"

MODEL_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert PROJECT_ROOT.exists(), (
    "Project root is unavailable. Phase 00 must be completed before Phase 01."
)


In [6]:
def set_global_seed(seed: int) -> None:
    """Initialize deterministic pseudo-random states for the audit."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(True, warn_only=True)

    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


set_global_seed(BASE_SEED)

assert torch.cuda.is_available(), "CUDA GPU is required for the project audit."
DEVICE = torch.device("cuda")


## Resolve and lock the exact TTM revision

The official model selector is first queried in **key-only mode**. This separates model selection from model loading and makes the resolved revision explicit.

The same revision is then passed back to the loader through `model_revision`, preventing a later selector change from silently loading different weights.


In [7]:
resolved_model_revision = get_model(
    model_path=TTM_REPOSITORY,
    model_name="ttm",
    context_length=REQUESTED_CONTEXT_LENGTH,
    prediction_length=REQUESTED_PREDICTION_LENGTH,
    freq=DATA_FREQUENCY,
    prefer_l1_loss=PREFER_L1_LOSS,
    return_model_key=True,
)

assert isinstance(resolved_model_revision, str) and resolved_model_revision.strip(), (
    "The TTM selector did not return a valid model revision."
)

print("Resolved TTM revision:", resolved_model_revision)


INFO:/usr/local/lib/python3.13/dist-packages/tsfm_public/toolkit/get_model.py:Loading model from: ibm-granite/granite-timeseries-ttm-r2
INFO:/usr/local/lib/python3.13/dist-packages/tsfm_public/toolkit/get_model.py:Model path type is 2
INFO:/usr/local/lib/python3.13/dist-packages/tsfm_public/toolkit/get_model.py:Would load from ibm-granite/granite-timeseries-ttm-r2, revision = 512-48-ft-r2.1.


Resolved TTM revision: 512-48-ft-r2.1


In [8]:
hf_api = HfApi()
model_repository_info = hf_api.model_info(
    repo_id=TTM_REPOSITORY,
    revision=resolved_model_revision,
)

resolved_commit_sha = model_repository_info.sha

assert isinstance(resolved_commit_sha, str) and len(resolved_commit_sha) >= 7

print("Resolved Hugging Face commit SHA:", resolved_commit_sha)


Resolved Hugging Face commit SHA: b972f0c22190b7502764526004d16e2b4ed39e8c


In [9]:
ttm_model = get_model(
    model_path=TTM_REPOSITORY,
    model_name="ttm",
    context_length=REQUESTED_CONTEXT_LENGTH,
    prediction_length=REQUESTED_PREDICTION_LENGTH,
    freq=DATA_FREQUENCY,
    prefer_l1_loss=PREFER_L1_LOSS,
    model_revision=resolved_model_revision,
)

ttm_model = ttm_model.to(DEVICE)
ttm_model.eval()

model_class_name = type(ttm_model).__name__
model_module_name = type(ttm_model).__module__

print("Loaded model class:", model_class_name)
print("Loaded model module:", model_module_name)


INFO:/usr/local/lib/python3.13/dist-packages/tsfm_public/toolkit/get_model.py:Loading model from: ibm-granite/granite-timeseries-ttm-r2
INFO:/usr/local/lib/python3.13/dist-packages/tsfm_public/toolkit/get_model.py:Model path type is 2
INFO:/usr/local/lib/python3.13/dist-packages/tsfm_public/toolkit/get_model.py:Attempting model load from ibm-granite/granite-timeseries-ttm-r2, revision = 512-48-ft-r2.1.


config.json:   0%|          | 0.00/1.65k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.22MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/139 [00:00<?, ?it/s]

INFO:/usr/local/lib/python3.13/dist-packages/tsfm_public/toolkit/get_model.py:Model loaded successfully from ibm-granite/granite-timeseries-ttm-r2, revision = 512-48-ft-r2.1.
INFO:/usr/local/lib/python3.13/dist-packages/tsfm_public/toolkit/get_model.py:[TTM] context_length = 512, prediction_length = 48


Loaded model class: TinyTimeMixerForPrediction
Loaded model module: tsfm_public.models.tinytimemixer.modeling_tinytimemixer


## Configuration contract

The requested contract and the **actual loaded model configuration** are stored separately.

This distinction matters because the selected pre-trained TTM may internally have a longer native prediction length and apply a prediction filter to return the requested 28-day horizon.


In [10]:
def make_json_serializable(value: Any) -> Any:
    """Convert nested configuration objects into JSON-compatible values."""
    if value is None or isinstance(value, (str, int, float, bool)):
        return value

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, torch.dtype):
        return str(value)

    if isinstance(value, dict):
        return {
            str(key): make_json_serializable(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [make_json_serializable(item) for item in value]

    if is_dataclass(value):
        return make_json_serializable(asdict(value))

    if hasattr(value, "to_dict"):
        try:
            return make_json_serializable(value.to_dict())
        except Exception:
            pass

    return str(value)


model_config_dict = make_json_serializable(ttm_model.config.to_dict())

requested_model_contract = {
    "repository": TTM_REPOSITORY,
    "resolved_revision": resolved_model_revision,
    "resolved_commit_sha": resolved_commit_sha,
    "context_length": REQUESTED_CONTEXT_LENGTH,
    "prediction_length": REQUESTED_PREDICTION_LENGTH,
    "frequency": DATA_FREQUENCY,
    "prefer_l1_loss": PREFER_L1_LOSS,
    "seed": BASE_SEED,
}

actual_model_contract = {
    "model_class": model_class_name,
    "model_module": model_module_name,
    "config_context_length": getattr(ttm_model.config, "context_length", None),
    "config_prediction_length": getattr(ttm_model.config, "prediction_length", None),
    "config_prediction_filter_length": getattr(
        ttm_model.config, "prediction_filter_length", None
    ),
    "config_patch_length": getattr(ttm_model.config, "patch_length", None),
    "config_patch_stride": getattr(ttm_model.config, "patch_stride", None),
    "config_d_model": getattr(ttm_model.config, "d_model", None),
    "config_decoder_d_model": getattr(ttm_model.config, "decoder_d_model", None),
    "config_decoder_num_layers": getattr(
        ttm_model.config, "decoder_num_layers", None
    ),
    "config_loss": getattr(ttm_model.config, "loss", None),
    "config_resolution_prefix_tuning": getattr(
        ttm_model.config, "resolution_prefix_tuning", False
    ),
}

pd.DataFrame(
    [
        {"contract": "requested", **requested_model_contract},
        {"contract": "actual", **actual_model_contract},
    ]
)


,contract,repository,resolved_revision,resolved_commit_sha,context_length,prediction_length,frequency,prefer_l1_loss,seed,model_class,...,config_context_length,config_prediction_length,config_prediction_filter_length,config_patch_length,config_patch_stride,config_d_model,config_decoder_d_model,config_decoder_num_layers,config_loss,config_resolution_prefix_tuning
0,requested,ibm-granite/granite-timeseries-ttm-r2,512-48-ft-r2.1,b972f0c22190b7502764526004d16e2b4ed39e8c,512.0,28.0,D,False,2026.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,actual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TinyTimeMixerForPrediction,...,512.0,48.0,28.0,64.0,64.0,192.0,128.0,2.0,mse,True


## Parameter inventory

For a parameter tensor $\theta_i$ with $|\theta_i|$ scalar parameters, the total parameter count is:

$$
P_{\mathrm{total}}
=
\sum_i |\theta_i|
$$

The audit records every named parameter and module so that the later head-probing notebook can freeze and unfreeze components using **observed implementation paths**, not guessed names.


In [11]:
parameter_records: list[dict[str, Any]] = []

for parameter_name, parameter in ttm_model.named_parameters():
    parameter_records.append(
        {
            "parameter_name": parameter_name,
            "shape": list(parameter.shape),
            "num_parameters": int(parameter.numel()),
            "requires_grad_at_load": bool(parameter.requires_grad),
            "dtype": str(parameter.dtype),
        }
    )

parameter_table = pd.DataFrame(parameter_records)

total_parameter_count = int(parameter_table["num_parameters"].sum())
loaded_trainable_parameter_count = int(
    parameter_table.loc[
        parameter_table["requires_grad_at_load"], "num_parameters"
    ].sum()
)

print(f"Total parameters: {total_parameter_count:,}")
print(f"Trainable at initial load: {loaded_trainable_parameter_count:,}")


Total parameters: 799,756
Trainable at initial load: 799,756


In [12]:
module_records: list[dict[str, Any]] = []

for module_name, module in ttm_model.named_modules():
    direct_parameter_count = sum(
        parameter.numel()
        for parameter in module.parameters(recurse=False)
    )
    recursive_parameter_count = sum(
        parameter.numel()
        for parameter in module.parameters(recurse=True)
    )

    module_records.append(
        {
            "module_name": module_name if module_name else "<root>",
            "module_class": type(module).__name__,
            "direct_parameter_count": int(direct_parameter_count),
            "recursive_parameter_count": int(recursive_parameter_count),
            "depth": 0 if not module_name else module_name.count(".") + 1,
        }
    )

module_table = pd.DataFrame(module_records)

architecture_keyword_mask = module_table["module_name"].str.lower().str.contains(
    r"backbone|decoder|head", regex=True
)

architecture_module_candidates = module_table.loc[
    architecture_keyword_mask
].copy()

architecture_module_candidates


,module_name,module_class,direct_parameter_count,recursive_parameter_count,depth
1,backbone,TinyTimeMixerModel,0,552938,1
2,backbone.encoder,TinyTimeMixerEncoder,0,552938,2
3,backbone.encoder.patcher,Linear,12480,12480,3
4,backbone.encoder.mlp_mixer_encoder,TinyTimeMixerBlock,0,506730,3
5,backbone.encoder.mlp_mixer_encoder.mixers,ModuleList,0,506730,4
...,...,...,...,...,...
190,decoder.decoder_block.mixers.1.feature_mixer.g...,Linear,16512,16512,7
191,head,TinyTimeMixerForPredictionHead,0,55344,1
192,head.dropout_layer,Dropout,0,0,2
193,head.base_forecast_block,Linear,55344,55344,2


## Frequency-prefix contract

The selected TTM checkpoint may use **Resolution Prefix Tuning (RPT)**. When RPT is enabled, the model's forward pass requires a frequency token in addition to the historical input.

For the project's approved daily frequency,

$$
f = \mathrm{D},
$$

IBM's official frequency mapping gives:

$$
\tau_f = 8.
$$

For batch size $B$, the frequency-token tensor is:

$$
\mathbf{t}_f
=
[8,\ldots,8]
\in
\mathbb{N}^{B}.
$$

The implementation resolves this value from `DEFAULT_FREQUENCY_MAPPING`; it is not independently hard-coded into the model call.


In [13]:
resolution_prefix_tuning_enabled = bool(
    getattr(ttm_model.config, "resolution_prefix_tuning", False)
)

frequency_token_value = DEFAULT_FREQUENCY_MAPPING.get(DATA_FREQUENCY)

if resolution_prefix_tuning_enabled:
    assert frequency_token_value is not None, (
        f"No IBM frequency token is defined for frequency={DATA_FREQUENCY!r}."
    )

print("Resolution Prefix Tuning enabled:", resolution_prefix_tuning_enabled)
print("Frequency:", DATA_FREQUENCY)
print("Resolved frequency token:", frequency_token_value)


Resolution Prefix Tuning enabled: True
Frequency: D
Resolved frequency token: 8


## Empirical forward-interface audit

The synthetic input is used only to inspect model interfaces. It is not data and does not contribute to any reported forecasting result.

Forward hooks record tensor shapes at implementation modules whose names contain `backbone`, `decoder`, or `head`. The model output is independently summarized so that internal hooks and public return fields can be cross-checked.


In [14]:
def tensor_shape_summary(value: Any) -> Any:
    """Recursively summarize tensors without retaining their numerical contents."""
    if torch.is_tensor(value):
        return {
            "type": "tensor",
            "shape": list(value.shape),
            "dtype": str(value.dtype),
            "device": str(value.device),
        }

    if isinstance(value, dict):
        return {
            str(key): tensor_shape_summary(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [tensor_shape_summary(item) for item in value]

    if hasattr(value, "to_tuple"):
        try:
            return tensor_shape_summary(value.to_tuple())
        except Exception:
            pass

    return {"type": type(value).__name__}


def public_output_summary(model_output: Any) -> dict[str, Any]:
    """Summarize public output fields exposed by a Hugging Face ModelOutput."""
    summary: dict[str, Any] = {}

    if hasattr(model_output, "keys"):
        for key in model_output.keys():
            summary[str(key)] = tensor_shape_summary(getattr(model_output, key))

    if not summary:
        summary["raw_output"] = tensor_shape_summary(model_output)

    return summary


In [15]:
hook_records: list[dict[str, Any]] = []
hook_handles = []

candidate_module_names = set(
    architecture_module_candidates["module_name"].tolist()
)

for module_name, module in ttm_model.named_modules():
    normalized_name = module_name if module_name else "<root>"

    if normalized_name not in candidate_module_names:
        continue

    def make_hook(captured_name: str):
        def audit_hook(module, inputs, output):
            hook_records.append(
                {
                    "module_name": captured_name,
                    "module_class": type(module).__name__,
                    "input_summary": tensor_shape_summary(inputs),
                    "output_summary": tensor_shape_summary(output),
                }
            )
        return audit_hook

    hook_handles.append(
        module.register_forward_hook(make_hook(normalized_name))
    )


AUDIT_BATCH_SIZE = 1

synthetic_past_values = torch.randn(
    AUDIT_BATCH_SIZE,
    REQUESTED_CONTEXT_LENGTH,
    1,
    dtype=torch.float32,
    device=DEVICE,
)

synthetic_frequency_token = None
if resolution_prefix_tuning_enabled:
    synthetic_frequency_token = torch.full(
        (AUDIT_BATCH_SIZE,),
        int(frequency_token_value),
        dtype=torch.long,
        device=DEVICE,
    )


def run_ttm_audit_forward(
    model: torch.nn.Module,
    past_values: torch.Tensor,
    frequency_token: torch.Tensor | None,
):
    """Execute an inference-only TTM forward pass using the checkpoint's actual RPT contract."""
    forward_kwargs: dict[str, Any] = {
        "past_values": past_values,
        "return_dict": True,
    }

    if frequency_token is not None:
        forward_kwargs["freq_token"] = frequency_token

    try:
        output = model(
            **forward_kwargs,
            output_hidden_states=True,
        )
        hidden_states_supported = True
    except TypeError as error:
        if "output_hidden_states" not in str(error):
            raise

        output = model(**forward_kwargs)
        hidden_states_supported = False

    return output, hidden_states_supported


try:
    with torch.no_grad():
        audit_output, output_hidden_states_supported = run_ttm_audit_forward(
            model=ttm_model,
            past_values=synthetic_past_values,
            frequency_token=synthetic_frequency_token,
        )
finally:
    for handle in hook_handles:
        handle.remove()

model_output_summary = public_output_summary(audit_output)

print("Model output fields:")
print(json.dumps(model_output_summary, indent=2))


Model output fields:
{
  "prediction_outputs": {
    "type": "tensor",
    "shape": [
      1,
      28,
      1
    ],
    "dtype": "torch.float32",
    "device": "cuda:0"
  },
  "backbone_hidden_state": {
    "type": "tensor",
    "shape": [
      1,
      1,
      9,
      192
    ],
    "dtype": "torch.float32",
    "device": "cuda:0"
  },
  "decoder_hidden_state": {
    "type": "tensor",
    "shape": [
      1,
      1,
      9,
      128
    ],
    "dtype": "torch.float32",
    "device": "cuda:0"
  },
  "hidden_states": [
    {
      "type": "tensor",
      "shape": [
        1,
        1,
        9,
        192
      ],
      "dtype": "torch.float32",
      "device": "cuda:0"
    },
    {
      "type": "tensor",
      "shape": [
        1,
        1,
        36,
        48
      ],
      "dtype": "torch.float32",
      "device": "cuda:0"
    },
    {
      "type": "tensor",
      "shape": [
        1,
        1,
        36,
        48
      ],
      "dtype": "torch.float32",
   

In [16]:
hook_table = pd.DataFrame(hook_records)

if hook_table.empty:
    raise RuntimeError(
        "No backbone/decoder/head forward hooks were captured. "
        "The implementation interface requires manual review before proceeding."
    )

hook_table[["module_name", "module_class"]]


,module_name,module_class
0,backbone.scaler,TinyTimeMixerStdScaler
1,backbone.patching,TinyTimeMixerPatchify
2,backbone.encoder.patcher,Linear
3,backbone.encoder.freq_mod.0,Embedding
4,backbone.encoder.freq_mod.1,Linear
...,...,...
184,decoder,TinyTimeMixerDecoder
185,head.flatten,Flatten
186,head.dropout_layer,Dropout
187,head.base_forecast_block,Linear


## Forecast-output contract

The audit extracts tensor fields from the model output and identifies all tensors whose dimensions contain the requested 28-step horizon.

No field is renamed to `base_forecast` until its observed shape confirms that it is a forecasting output.


In [17]:
def flatten_tensor_summaries(
    summary: Any,
    prefix: str = "",
) -> list[dict[str, Any]]:
    """Flatten nested shape summaries into a searchable table."""
    records: list[dict[str, Any]] = []

    if isinstance(summary, dict) and summary.get("type") == "tensor":
        records.append(
            {
                "path": prefix,
                "shape": summary["shape"],
                "dtype": summary["dtype"],
                "device": summary["device"],
            }
        )
        return records

    if isinstance(summary, dict):
        for key, value in summary.items():
            child_prefix = f"{prefix}.{key}" if prefix else str(key)
            records.extend(flatten_tensor_summaries(value, child_prefix))
        return records

    if isinstance(summary, list):
        for index, value in enumerate(summary):
            child_prefix = f"{prefix}[{index}]"
            records.extend(flatten_tensor_summaries(value, child_prefix))

    return records


output_tensor_records = flatten_tensor_summaries(model_output_summary)
output_tensor_table = pd.DataFrame(output_tensor_records)

if output_tensor_table.empty:
    raise RuntimeError("The TTM forward pass exposed no tensor output.")

output_tensor_table["contains_requested_horizon"] = output_tensor_table["shape"].apply(
    lambda shape: REQUESTED_PREDICTION_LENGTH in shape
)

output_tensor_table


,path,shape,dtype,device,contains_requested_horizon
0,prediction_outputs,"[1, 28, 1]",torch.float32,cuda:0,True
1,backbone_hidden_state,"[1, 1, 9, 192]",torch.float32,cuda:0,False
2,decoder_hidden_state,"[1, 1, 9, 128]",torch.float32,cuda:0,False
3,hidden_states[0],"[1, 1, 9, 192]",torch.float32,cuda:0,False
4,hidden_states[1],"[1, 1, 36, 48]",torch.float32,cuda:0,False
5,hidden_states[2],"[1, 1, 36, 48]",torch.float32,cuda:0,False
6,hidden_states[3],"[1, 1, 36, 48]",torch.float32,cuda:0,False
7,hidden_states[4],"[1, 1, 9, 192]",torch.float32,cuda:0,False
8,hidden_states[5],"[1, 1, 9, 192]",torch.float32,cuda:0,False
9,hidden_states[6],"[1, 1, 18, 96]",torch.float32,cuda:0,False


## Candidate component contract

The architecture requires three implementation components in the original TTM path:

1. pre-trained TTM backbone;
2. Slim TTM Decoder;
3. Forecasting Head.

This phase does not yet freeze parameters for head probing. Instead, it records the actual module paths that match these architectural roles and saves the full module inventory for review.


In [18]:
def keyword_candidates(keyword: str) -> list[str]:
    mask = module_table["module_name"].str.lower().str.contains(
        keyword.lower(),
        regex=False,
    )
    return module_table.loc[mask, "module_name"].tolist()


backbone_candidates = keyword_candidates("backbone")
decoder_candidates = keyword_candidates("decoder")
head_candidates = keyword_candidates("head")

component_candidate_summary = {
    "backbone_candidates": backbone_candidates,
    "decoder_candidates": decoder_candidates,
    "head_candidates": head_candidates,
}

print(json.dumps(component_candidate_summary, indent=2))


{
  "backbone_candidates": [
    "backbone",
    "backbone.encoder",
    "backbone.encoder.patcher",
    "backbone.encoder.mlp_mixer_encoder",
    "backbone.encoder.mlp_mixer_encoder.mixers",
    "backbone.encoder.mlp_mixer_encoder.mixers.0",
    "backbone.encoder.mlp_mixer_encoder.mixers.0.mixer_layers",
    "backbone.encoder.mlp_mixer_encoder.mixers.0.mixer_layers.0",
    "backbone.encoder.mlp_mixer_encoder.mixers.0.mixer_layers.0.patch_mixer",
    "backbone.encoder.mlp_mixer_encoder.mixers.0.mixer_layers.0.patch_mixer.norm",
    "backbone.encoder.mlp_mixer_encoder.mixers.0.mixer_layers.0.patch_mixer.norm.norm",
    "backbone.encoder.mlp_mixer_encoder.mixers.0.mixer_layers.0.patch_mixer.mlp",
    "backbone.encoder.mlp_mixer_encoder.mixers.0.mixer_layers.0.patch_mixer.mlp.fc1",
    "backbone.encoder.mlp_mixer_encoder.mixers.0.mixer_layers.0.patch_mixer.mlp.dropout1",
    "backbone.encoder.mlp_mixer_encoder.mixers.0.mixer_layers.0.patch_mixer.mlp.fc2",
    "backbone.encoder.mlp_mixer_e

In [19]:
assert backbone_candidates, "No backbone module path was discovered."
assert decoder_candidates, "No decoder module path was discovered."
assert head_candidates, "No forecasting-head module path was discovered."


## Persistent model snapshot

The exact loaded model is stored in Google Drive after revision resolution. Later notebooks will use this project-local snapshot rather than asking the remote selector to choose a model again.

A reload check verifies that the saved snapshot preserves the same model class and requested 28-step output behavior.


In [20]:
# Save a clean CPU snapshot to persistent storage.
ttm_model_cpu = copy.deepcopy(ttm_model).to("cpu")
ttm_model_cpu.save_pretrained(MODEL_SNAPSHOT_DIR)

snapshot_config_path = MODEL_SNAPSHOT_DIR / "config.json"
assert snapshot_config_path.exists(), "Saved TTM snapshot is missing config.json."

snapshot_files = sorted(
    path.name
    for path in MODEL_SNAPSHOT_DIR.iterdir()
    if path.is_file()
)

print("Saved model snapshot files:", snapshot_files)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model snapshot files: ['config.json', 'model.safetensors']


In [21]:
model_class = type(ttm_model_cpu)

reloaded_model = model_class.from_pretrained(
    MODEL_SNAPSHOT_DIR,
).to(DEVICE)
reloaded_model.eval()

reloaded_rpt_enabled = bool(
    getattr(reloaded_model.config, "resolution_prefix_tuning", False)
)

reloaded_frequency_token = None
if reloaded_rpt_enabled:
    reloaded_frequency_token = torch.full(
        (AUDIT_BATCH_SIZE,),
        int(frequency_token_value),
        dtype=torch.long,
        device=DEVICE,
    )

with torch.no_grad():
    reloaded_output, _ = run_ttm_audit_forward(
        model=reloaded_model,
        past_values=synthetic_past_values,
        frequency_token=reloaded_frequency_token,
    )

reloaded_output_summary = public_output_summary(reloaded_output)
reloaded_tensor_records = flatten_tensor_summaries(reloaded_output_summary)

assert reloaded_tensor_records, "Reloaded TTM exposed no tensor outputs."

reload_horizon_verified = any(
    REQUESTED_PREDICTION_LENGTH in record["shape"]
    for record in reloaded_tensor_records
)

assert reload_horizon_verified, (
    "Reloaded model did not expose a tensor containing the requested 28-step horizon."
)

del reloaded_model
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/139 [00:00<?, ?it/s]

## Save audit artifacts

The audit produces machine-readable model provenance and human-inspectable tables. These artifacts become dependencies of the later head-probing and decoder-cross-feed phases.


In [22]:
parameter_table_path = RESULTS_DIR / "01_TTM_PARAMETER_INVENTORY.csv"
module_table_path = RESULTS_DIR / "01_TTM_MODULE_INVENTORY.csv"
hook_table_path = RESULTS_DIR / "01_TTM_INTERFACE_HOOKS.json"
contract_path = METADATA_DIR / "01_TTM_MODEL_CONTRACT.json"
config_path = METADATA_DIR / "01_TTM_FULL_CONFIG.json"

parameter_table.to_csv(parameter_table_path, index=False)
module_table.to_csv(module_table_path, index=False)

hook_table_path.write_text(
    json.dumps(hook_records, indent=2),
    encoding="utf-8",
)

config_path.write_text(
    json.dumps(model_config_dict, indent=2),
    encoding="utf-8",
)

audit_contract = {
    "status": "PHASE_01_TTM_CONTRACT_AUDITED",
    "granite_tsfm_distribution_version": TSFM_DISTRIBUTION_VERSION,
    "tsfm_source_resolution": tsfm_source_resolution,
    "tsfm_source_commit": tsfm_source_commit,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "requested_contract": requested_model_contract,
    "actual_contract": actual_model_contract,
    "output_hidden_states_supported": output_hidden_states_supported,
    "resolution_prefix_tuning_enabled": resolution_prefix_tuning_enabled,
    "frequency": DATA_FREQUENCY,
    "frequency_token_value": frequency_token_value,
    "total_parameter_count": total_parameter_count,
    "trainable_parameter_count_at_load": loaded_trainable_parameter_count,
    "component_candidates": component_candidate_summary,
    "public_model_output": model_output_summary,
    "hook_records": hook_records,
    "snapshot_directory": str(MODEL_SNAPSHOT_DIR),
    "snapshot_files": snapshot_files,
    "reload_horizon_verified": reload_horizon_verified,
}

contract_payload = json.dumps(
    audit_contract,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

audit_contract["contract_sha256"] = hashlib.sha256(contract_payload).hexdigest()

contract_path.write_text(
    json.dumps(audit_contract, indent=2),
    encoding="utf-8",
)


148491

In [23]:
status_path = PROJECT_ROOT / "PROJECT_STATUS.md"

status_text = f"""# EA-TTM Project Status

## Current stage
Phase 01 — TTM model contract and interface audit completed.

## Persistent project root
`{PROJECT_ROOT}`

## Locked TTM contract
- Repository: `{TTM_REPOSITORY}`
- Resolved revision: `{resolved_model_revision}`
- Hugging Face commit SHA: `{resolved_commit_sha}`
- Requested context length: `{REQUESTED_CONTEXT_LENGTH}`
- Requested forecast horizon: `{REQUESTED_PREDICTION_LENGTH}`
- Frequency: `{DATA_FREQUENCY}`
- MSE preference: `{not PREFER_L1_LOSS}`

## Model snapshot
`{MODEL_SNAPSHOT_DIR}`

## Model class
`{model_module_name}.{model_class_name}`

## Parameter count
- Total: `{total_parameter_count}`
- Trainable at initial load: `{loaded_trainable_parameter_count}`

## Next phase
Audit the M5 dataset contract, exact raw files, chronological split boundaries,
forecast origins, and future-known retail covariates before any model evaluation.
"""

status_path.write_text(status_text, encoding="utf-8")


901

## Phase validation

Phase 01 is complete only if all of the following hold:

$$
\begin{aligned}
&\text{exact TTM revision resolved}
\\
&\land\ \text{commit SHA recorded}
\\
&\land\ \text{model loaded successfully}
\\
&\land\ \text{backbone candidate observed}
\\
&\land\ \text{decoder candidate observed}
\\
&\land\ \text{head candidate observed}
\\
&\land\ \text{28-step forecast tensor observed}
\\
&\land\ \text{persistent model snapshot reload verified}
\end{aligned}
$$


In [24]:
phase_validation = {
    "resolved_revision": bool(resolved_model_revision),
    "resolved_commit_sha": bool(resolved_commit_sha),
    "backbone_candidates_found": bool(backbone_candidates),
    "decoder_candidates_found": bool(decoder_candidates),
    "head_candidates_found": bool(head_candidates),
    "requested_horizon_observed": bool(
        output_tensor_table["contains_requested_horizon"].any()
    ),
    "snapshot_reload_verified": bool(reload_horizon_verified),
    "contract_artifact_exists": contract_path.exists(),
}

assert all(phase_validation.values()), (
    f"Phase 01 validation failed: {phase_validation}"
)

phase_validation


{'resolved_revision': True,
 'resolved_commit_sha': True,
 'backbone_candidates_found': True,
 'decoder_candidates_found': True,
 'head_candidates_found': True,
 'requested_horizon_observed': True,
 'snapshot_reload_verified': True,
 'contract_artifact_exists': True}

## Return block

The return block contains only the evidence needed to decide the next implementation step. The complete tensor and module details remain stored in the audit artifacts.


In [25]:
def compact_hook_shape_records(
    records: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    return [
        {
            "module_name": record["module_name"],
            "module_class": record["module_class"],
            "output_summary": record["output_summary"],
        }
        for record in records
    ]


return_block = {
    "status": "PHASE_01_COMPLETE",
    "repository": TTM_REPOSITORY,
    "granite_tsfm_distribution_version": TSFM_DISTRIBUTION_VERSION,
    "tsfm_source_resolution": tsfm_source_resolution,
    "tsfm_source_commit": tsfm_source_commit,
    "resolved_revision": resolved_model_revision,
    "resolved_commit_sha": resolved_commit_sha,
    "requested_context_length": REQUESTED_CONTEXT_LENGTH,
    "requested_prediction_length": REQUESTED_PREDICTION_LENGTH,
    "frequency": DATA_FREQUENCY,
    "prefer_l1_loss": PREFER_L1_LOSS,
    "resolution_prefix_tuning_enabled": resolution_prefix_tuning_enabled,
    "frequency_token_value": frequency_token_value,
    "model_class": model_class_name,
    "actual_config": actual_model_contract,
    "total_parameters": total_parameter_count,
    "trainable_parameters_at_load": loaded_trainable_parameter_count,
    "model_output_summary": model_output_summary,
    "component_candidates": component_candidate_summary,
    "interface_hook_summary": compact_hook_shape_records(hook_records),
    "snapshot_reload_verified": reload_horizon_verified,
    "contract_sha256": audit_contract["contract_sha256"],
    "artifacts": {
        "model_contract": str(contract_path),
        "full_model_config": str(config_path),
        "parameter_inventory": str(parameter_table_path),
        "module_inventory": str(module_table_path),
        "interface_hooks": str(hook_table_path),
        "model_snapshot": str(MODEL_SNAPSHOT_DIR),
        "project_status": str(status_path),
    },
}

print("PHASE 01 RETURN BLOCK")
print(json.dumps(return_block, indent=2))


PHASE 01 RETURN BLOCK
{
  "status": "PHASE_01_COMPLETE",
  "repository": "ibm-granite/granite-timeseries-ttm-r2",
  "granite_tsfm_distribution_version": "0.3.9",
  "tsfm_source_resolution": "installed_distribution",
  "tsfm_source_commit": null,
  "resolved_revision": "512-48-ft-r2.1",
  "resolved_commit_sha": "b972f0c22190b7502764526004d16e2b4ed39e8c",
  "requested_context_length": 512,
  "requested_prediction_length": 28,
  "frequency": "D",
  "prefer_l1_loss": false,
  "resolution_prefix_tuning_enabled": true,
  "frequency_token_value": 8,
  "model_class": "TinyTimeMixerForPrediction",
  "actual_config": {
    "model_class": "TinyTimeMixerForPrediction",
    "model_module": "tsfm_public.models.tinytimemixer.modeling_tinytimemixer",
    "config_context_length": 512,
    "config_prediction_length": 48,
    "config_prediction_filter_length": 28,
    "config_patch_length": 64,
    "config_patch_stride": 64,
    "config_d_model": 192,
    "config_decoder_d_model": 128,
    "config_decode